# 🎬 AutoVSF - Google Colab Edition

Automated hard subtitle (hardsub) extraction tool using VideoSubFinder and Google Drive OCR.

🔗 **Repository:** [https://github.com/lionc2240/autovsf-colab](https://github.com/lionc2240/autovsf-colab)

## 🚀 Step 1: Connect to Google Drive
We will save all code and results on Drive to prevent data loss when Colab shuts down.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create working directory on Drive
WORKDIR = "/content/drive/MyDrive/AutoVSF"
if not os.path.exists(WORKDIR):
    os.makedirs(WORKDIR)
    print(f"✅ Working directory created at: {WORKDIR}")

%cd $WORKDIR

## 🛠️ Step 2: Environment Setup
Automatically download source code and install necessary system libraries (VideoSubFinder, Xvfb, ffmpeg...)

In [ ]:
# 1. Clone or update source code
import os
if os.path.exists("headless.py"):
    print("✅ Already in repository.")
    !git pull
elif os.path.exists("autovsf-colab"):
    %cd autovsf-colab
    print("✅ Repository found. Updating...")
    !git pull
else:
    print("📥 Cloning repository...")
    !git clone https://github.com/lionc2240/autovsf-colab.git
    %cd autovsf-colab

# 2. Run system setup script (Handles legacy libraries for Colab Ubuntu 22.04)
# Note: install.sh will automatically handle missing .deb files
!chmod +x install.sh
!./install.sh

## 🔑 Step 3: Google Cloud Configuration (OCR)
1. Upload your `credentials.json` file to the `AutoVSF/autovsf-colab/` folder on Google Drive.
2. If this is the first run, the tool will ask you to authenticate via a link.

In [ ]:
if not os.path.exists("credentials.json"):
    print("❌ ERROR: 'credentials.json' file not found!")
    print("👉 Please upload this file to the AutoVSF/autovsf-colab/ folder on your Drive and run this cell again.")
else:
    print("✅ credentials.json found. Ready!")

## 🎬 Step 4: Launch Subtitle Extraction
Choose between CLI mode (Step 4.1) or Visual Mode (Step 4.2).

In [ ]:
# @title 🚀 Step 4.1: CLI Mode (Fast Execution)
# @markdown ### 💡 Hướng dẫn chọn vùng Crop:
# @markdown - Các thông số chạy từ **0.0** đến **1.0** (tương ứng 0% - 100% kích thước video).
# @markdown - **Trục Y (Dọc):** **0.0 là Đáy**, **1.0 là Đỉnh**.
# @markdown   - *Ví dụ:* Để quét 30% phần dưới video (vùng phụ đề), đặt **Bottom = 0.0** và **Top = 0.3**.
# @markdown - **Trục X (Ngang):** 0.0 là bên trái, 1.0 là bên phải.

VIDEO_PATH = "/content/video_test.mp4" # @param {type:"string"}
TOP_CROP = 0.3 # @param {type:"number"}
BOTTOM_CROP = 0.0 # @param {type:"number"}
LEFT_CROP = 0.0 # @param {type:"number"}
RIGHT_CROP = 1.0 # @param {type:"number"}

!python3 headless.py "$VIDEO_PATH" --top $TOP_CROP --bottom $BOTTOM_CROP --left $LEFT_CROP --right $RIGHT_CROP

In [ ]:
# @title 🎨 Step 4.2: Visual Mode (Gradio UI + Profile Management)
try:
    import gradio as gr
except:
    !pip install -q gradio
    import gradio as gr

import os
import threading
import time
from pathlib import Path
import config as C
import headless

def load_last_settings():
    cfg = C.load()
    profile = cfg.get("crop_profiles", {}).get("colab_last_run", {})
    return (
        profile.get("top", 0.3), 
        profile.get("bottom", 0.0), 
        profile.get("left", 0.0), 
        profile.get("right", 1.0)
    )

def save_settings(top, bottom, left, right):
    cfg = C.load()
    if "crop_profiles" not in cfg: cfg["crop_profiles"] = {}
    cfg["crop_profiles"]["colab_last_run"] = {
        "top": top, "bottom": bottom, "left": left, "right": right
    }
    C.save(cfg)
    return "✅ Profile saved to settings.json"

def start_extraction(video_path, top, bottom, left, right, progress=gr.Progress()):
    if not video_path or not os.path.exists(video_path):
        return "❌ Video path invalid."
    
    save_settings(top, bottom, left, right)
    C.state.reset()
    
    def run_task():
        try:
            y_min, y_max = min(top, bottom), max(top, bottom)
            if y_min == y_max: y_max += 0.1
            headless.run_headless(video_path, y_max, y_min, left, right)
        except Exception as e: print(f"Error: {e}")
    
    t = threading.Thread(target=run_task, daemon=True)
    t.start()
    
    progress(0, desc="🚀 Initializing...")
    while t.is_alive():
        if C.state.total > 0:
            prog = C.state.done / C.state.total
            progress(prog, desc=f"⏳ OCR Progress: {C.state.done}/{C.state.total}")
        else:
            progress(0, desc="🎞️ Scanning video...")
        time.sleep(1)
    
    if C.state.total > 0 and C.state.done >= C.state.total:
        return f"✅ Done! Output: {Path(video_path).with_suffix('.srt')}"
    return "⚠️ Finished. Check logs if file is missing."

def get_status_text():
    if C.state.total == 0: return "### Status: Ready"
    prog = C.state.done / C.state.total if C.state.total > 0 else 0
    txt = f"### Progress: {C.state.done}/{C.state.total} ({prog*100:.1f}%)"
    if C.state.done >= C.state.total: txt += "\n✅ Done!"
    return txt

with gr.Blocks(title="AutoVSF Visual") as demo:
    gr.Markdown("# 🎬 AutoVSF Visual Interface")
    gr.Markdown("**Coordinate System:** 0.0 = Bottom, 1.0 = Top. Adjust sliders to select subtitle area.")
    
    with gr.Row():
        with gr.Column(scale=2):
            video_in = gr.Textbox(label="Video Path", value="/content/video_test.mp4")
            with gr.Group():
                gr.Markdown("### ✂️ Crop Settings (Relative 0.0 - 1.0)")
                with gr.Row():
                    t_sld = gr.Slider(0, 1, value=0.3, step=0.01, label="Top (Upper boundary)")
                    b_sld = gr.Slider(0, 1, value=0.0, step=0.01, label="Bottom (Lower boundary)")
                with gr.Row():
                    l_sld = gr.Slider(0, 1, value=0.0, step=0.01, label="Left")
                    r_sld = gr.Slider(0, 1, value=1.0, step=0.01, label="Right")
            with gr.Row():
                btn_save = gr.Button("💾 Save Profile")
                btn_start = gr.Button("🚀 Start Extraction", variant="primary")
        
        with gr.Column(scale=1):
            st_out = gr.Textbox(label="System Status", interactive=False)
            prog_txt = gr.Markdown("### Status: Ready")
            
            demo.load(load_last_settings, None, [t_sld, b_sld, l_sld, r_sld])
            
            if hasattr(gr, "Timer"):
                gr.Timer(2).tick(get_status_text, None, prog_txt)
            else:
                demo.load(get_status_text, None, prog_txt)
            
            
    btn_save.click(save_settings, [t_sld, b_sld, l_sld, r_sld], st_out)
    btn_start.click(start_extraction, [video_in, t_sld, b_sld, l_sld, r_sld], st_out)

demo.queue().launch(share=True, height=600)

## 🧹 Step 5: Cleanup (Optional)
Temporary images are stored in `/content` and will be auto-deleted when session ends.

In [ ]:
import os
from pathlib import Path

out_dir = f"/content/{Path(VIDEO_PATH).stem}_out"
if os.path.exists(out_dir):
    print(f"🧹 Removing temporary folder: {out_dir}")
    !rm -rf "$out_dir"
    print("✅ Cleanup complete.")